In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time
import xgboost as xgb

# Large synthetic dataset
np.random.seed(42)
X = np.random.rand(1_000_000, 100)
y = np.random.rand(1_000_000)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:

# GPU Training
params_gpu = {
    "tree_method": "hist",
    "device": 'cuda',  
    "max_depth": 8,
    "learning_rate": 0.05,
    "objective": "reg:squarederror",
}

dtrain_gpu = xgb.DMatrix(X_train_scaled, label=y_train)
dtest_gpu = xgb.DMatrix(X_test_scaled, label=y_test)

start_time = time.time()
model_gpu = xgb.train(params_gpu, dtrain_gpu, num_boost_round=1000)
gpu_train_time = time.time() - start_time

start_time = time.time()
train_pred_gpu = model_gpu.predict(dtrain_gpu)
test_pred_gpu = model_gpu.predict(dtest_gpu)
gpu_pred_time = time.time() - start_time

train_mae_gpu = mean_absolute_error(y_train, train_pred_gpu)
test_mae_gpu = mean_absolute_error(y_test, test_pred_gpu)

print(f"[GPU] Training Time: {gpu_train_time:.3f}s")
print(f"[GPU] Prediction Time: {gpu_pred_time:.3f}s")
print(f"[GPU] Train MAE: {train_mae_gpu:.6f}")
print(f"[GPU] Test MAE:  {test_mae_gpu:.6f}")


GPU Training Time: 41.123324394226074
GPU Prediction Time: 5.087620973587036
GPU Train MAE: 0.2177910655555475
GPU Test MAE: 0.25081917471725845


In [3]:

# CPU Training (for comparison)
from xgboost import XGBRegressor

model_cpu = XGBRegressor(
    n_estimators=1000,
    max_depth=8,
    learning_rate=0.05,
    tree_method="hist",
    device="cpu",              # Explicit CPU device
)

start_time = time.time()
model_cpu.fit(X_train_scaled, y_train)
cpu_train_time = time.time() - start_time

start_time = time.time()
train_pred_cpu = model_cpu.predict(X_train_scaled)
test_pred_cpu = model_cpu.predict(X_test_scaled)
cpu_pred_time = time.time() - start_time

train_mae_cpu = mean_absolute_error(y_train, train_pred_cpu)
test_mae_cpu = mean_absolute_error(y_test, test_pred_cpu)

print("CPU Training Time:", cpu_train_time)
print("CPU Prediction Time:", cpu_pred_time)
print("CPU Train MAE:", train_mae_cpu)
print("CPU Test MAE:", test_mae_cpu)


CPU Training Time: 95.76777863502502
CPU Prediction Time: 2.167520523071289
CPU Train MAE: 0.21728239964883847
CPU Test MAE: 0.2508184107651189


In [4]:

# --- Comparison Summary ---
import pandas as pd

summary = pd.DataFrame({
    "Metric": ["Training Time (s)", "Prediction Time (s)", "Train MAE", "Test MAE"],
    "GPU (CUDA)": [round(gpu_train_time, 3), round(gpu_pred_time, 3),
                   round(train_mae_gpu, 6), round(test_mae_gpu, 6)],
    "CPU":        [round(cpu_train_time, 3), round(cpu_pred_time, 3),
                   round(train_mae_cpu, 6), round(test_mae_cpu, 6)],
})
summary["Speedup (CPU/GPU)"] = (summary["CPU"] / summary["GPU (CUDA)"]).round(2)
print(summary.to_string(index=False))


             Metric  GPU (CUDA)       CPU  Speedup (CPU/GPU)
  Training Time (s)   41.123000 95.768000               2.33
Prediction Time (s)    5.088000  2.168000               0.43
          Train MAE    0.217791  0.217282               1.00
           Test MAE    0.250819  0.250818               1.00


#--------------------------------------------------------------
### Local 
             Metric  GPU (CUDA)       CPU  Speedup (CPU/GPU)
  Training Time (s)   41.123000 95.768000               2.33
Prediction Time (s)    5.088000  2.168000               0.43
#-----------------------------------------------------------------
### Colab - Base 
             Metric   GPU (CUDA)       CPU                    Speedup (CPU/GPU)
  Training Time (s)    -----           95.768000               2.33
Prediction Time (s)    ------          2.168000               0.43
